# 📦 Notebook 5 – Inventory Optimization
**RetailPulse | EOQ + Safety Stock + Reorder Point**

Target: Reduce overstock/understock by 25–40%


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0,'../..') 
plt.style.use('dark_background')
print('Libraries loaded ✅')

In [ ]:
from src.features.feature_engineering import build_product_daily_sales
df = pd.read_parquet('../../data/processed/retail_clean.parquet')
product_daily = build_product_daily_sales(df)
product_daily['Date'] = pd.to_datetime(product_daily['Date'])
print(f'Products tracked: {product_daily["StockCode"].nunique():,}')
product_daily.head()

In [ ]:
from src.models.inventory import (
    calculate_safety_stock, calculate_reorder_point,
    calculate_eoq, build_inventory_recommendations, inventory_kpi_summary
)

recs = build_inventory_recommendations(product_daily, forecast_by_product={})
kpis = inventory_kpi_summary(recs)
print('=== Inventory KPIs ===')
for k,v in kpis.items():
    print(f'  {k}: {v}')
recs.head(10)

In [ ]:
recs.to_parquet('../../data/processed/inventory_recs.parquet', index=False)
print('Saved inventory_recs.parquet ✅')

fig, axes = plt.subplots(1,2,figsize=(14,5))
status_counts = recs['Status'].value_counts()
colors_map = {'🔴 Reorder Now':'#ef4444','🟢 OK':'#10b981','🟡 Overstock Risk':'#f59e0b'}
axes[0].pie(status_counts, labels=status_counts.index, colors=[colors_map.get(s,'gray') for s in status_counts.index], autopct='%1.1f%%')
axes[0].set_title('Inventory Status Distribution')

top_reorder = recs[recs['Status']=='🔴 Reorder Now'].head(10)
axes[1].barh(top_reorder['Description'].str[:30], top_reorder['ReorderPoint'], color='#ef4444')
axes[1].set_title('Top 10 Products Needing Reorder')
axes[1].set_xlabel('Reorder Point (units)')
axes[1].invert_yaxis()
plt.tight_layout()
plt.savefig('../../reports/inventory_analysis.png', dpi=150, bbox_inches='tight')
plt.show()